Chunking + dataset class + dataloader  

SEQ_LEN = 32      # Maximum input context  
TARGET = variable # Remaining tokens until <EOS>  
STRIDE = 16

In [1]:
import pickle

In [ ]:
with open('encoded_train.pkl','rb') as f:
  encoded_train=pickle.load(f)

with open('encoded_test.pkl','rb') as f:
  encoded_test=pickle.load(f)

with open('encoded_valid.pkl','rb') as f:
  encoded_valid=pickle.load(f)

with open('vocab.pkl','rb') as f:
  vocab=pickle.load(f)

In [4]:
import torch
from torch.utils.data import Dataset

class DatasetPrep(Dataset):

    def __init__(self, data, vocab,
                 seq_length=32,
                 max_output_length=50,
                 stride=16):

        self.data = data
        self.vocab = vocab
        self.seq_length = seq_length
        self.max_output_length = max_output_length

        # Each element = (row_number, split_position)
        self.indices = []

        for row_idx, row in enumerate(data):
            for i in range(seq_length, len(row), stride):
                self.indices.append((row_idx, i))

    def prep_data(self, row, i):

        # ---------- Encoder ----------
        input_seq = row[i-self.seq_length:i]

        encoder_input = (
            [self.vocab["<BOS>"]]
            + input_seq
            + [self.vocab["<EOS>"]]
        )

        # ---------- Find newline ----------
        nearest_line_end = len(row)

        for j in range(i, len(row)):
            if row[j] == self.vocab["<NEWLINE>"]:
                nearest_line_end = j
                break

        end = min(i + self.max_output_length,
                  nearest_line_end + 1)

        labels = row[i:end]

        # ---------- Decoder ----------
        decoder_input = [self.vocab["<BOS>"]] + labels
        decoder_output = labels + [self.vocab["<EOS>"]]

        return (
            torch.tensor(encoder_input),
            torch.tensor(decoder_input),
            torch.tensor(decoder_output)
        )

    def __getitem__(self, idx):

        row_idx, i = self.indices[idx]

        row = self.data[row_idx]

        return self.prep_data(row, i)

    def __len__(self):
        return len(self.indices)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

PAD = vocab["<PAD>"]

def collate_fn(batch):

    encoder_inputs, decoder_inputs, decoder_outputs = zip(*batch)

    encoder_inputs = pad_sequence(
        encoder_inputs,
        batch_first=True,
        padding_value=PAD
    )

    decoder_inputs = pad_sequence(
        decoder_inputs,
        batch_first=True,
        padding_value=PAD
    )

    decoder_outputs = pad_sequence(
        decoder_outputs,
        batch_first=True,
        padding_value=PAD
    )

    return encoder_inputs, decoder_inputs, decoder_outputs

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = DatasetPrep(encoded_train, vocab)
valid_dataset = DatasetPrep(encoded_valid, vocab)
test_dataset = DatasetPrep(encoded_test, vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
encoder, decoder_in, decoder_out = next(iter(train_loader))

print(encoder.shape)
print(decoder_in.shape)
print(decoder_out.shape)